# CS116 - PhoBERT MRC Training
## Đề tài T11: Hệ thống đọc hiểu & trả lời câu hỏi tiếng Việt
**Team**: Trần Trọng Tấn (25210334), Nguyễn Quang Lâm (25210289)
**Model**: vinai/phobert-base-v2
**Dataset**: UIT-ViQuAD 2.0 (39,569 QA pairs)

⚠️ **GPU Setup**: This notebook supports both NVIDIA CUDA and AMD ROCm GPUs.
- For AMD RX 6700 XT: Runtime → Change runtime type → GPU

In [ ]:
# Setup environment
!pip install transformers datasets torch scikit-learn evaluate --quiet
!pip install torch --index-url https://download.pytorch.org/whl/rocm5.7  # For AMD GPU

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'ROCm: {torch.version.hip is not None}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set GPU environment variables (especially for AMD)
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '10.3.0'
os.environ['PYTORCH_HIP_ALLOC_CONF'] = 'max_split_size:128'

In [ ]:
# Load dataset
from dataset_loader import load_or_create_viquad, split_dataset_by_context

print('Loading UIT-ViQuAD 2.0...')
full_data = load_or_create_viquad('viquad_sample.json')
train_data, val_data, test_data = split_dataset_by_context(full_data)

print(f'Train contexts: {len(train_data["data"][0]["paragraphs"])}')
print(f'Val contexts: {len(val_data["data"][0]["paragraphs"])}')
print(f'Test contexts: {len(test_data["data"][0]["paragraphs"])}')

In [ ]:
# Configure training
from train_phobert_qa import train_phobert_qa

# Model configuration (optimized for AMD RX 6700 XT 12GB VRAM)
model, tokenizer, history = train_phobert_qa(
    train_data_path='viquad_sample.json',
    output_dir='/content/drive/MyDrive/cs116-phobert-viquad',
    model_name='vinai/phobert-base-v2',
    num_epochs=3,
    learning_rate=3e-5,
    batch_size=8,  # Reduced for 12GB VRAM compatibility
    warmup_ratio=0.1,
    weight_decay=0.01
)

In [ ]:
# Training with visualization
from train_phobert_qa import VietnameseQAModel

qa = VietnameseQAModel()
context = "Trường Đại học Công nghệ Thông tin được thành lập ngày 8 tháng 6 năm 2006."
question = "Trường thành lập khi nào?"

print('Prediction:', qa.predict_span(context, question))